# 3 — Run Serving Experiments & Collect Metrics

Five serving options covering all required optimisation dimensions:

| # | Option | Backend | Workers | Hardware | Optimisation |
|---|--------|---------|---------|----------|--------------|
| 1 | `baseline` | PyTorch `.pt` | 1 uvicorn | CPU | — (simplest real serving) |
| 2 | `onnx_cpu` | ONNX Runtime | 1 uvicorn | CPU | Model-level only |
| 3 | `torch_multiworker` | PyTorch `.pt` | 4 gunicorn | CPU | System-level only |
| 4 | `onnx_multiworker` | ONNX Runtime | 4 gunicorn | CPU | Model + System (combined) |
| 5 | `onnx_lb` | ONNX × 2 + nginx | 1 uvicorn × 2 | CPU | Infrastructure-level |

**Prerequisites:** notebook 2 completed, server up, repo cloned, model files in place.

## Configuration

In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease
import os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

In [ ]:
# runs in Chameleon Jupyter environment
NET_ID     = "YOUR_NET_ID"
LEASE_NAME = f"serve_jellyfin_{NET_ID}"
REPO_DIR   = "~/jellyfin-recommender"

TOTAL_REQUESTS   = 500
CONCURRENCY_LOW  = 1   # isolates per-request latency
CONCURRENCY_HIGH = 8   # stresses multi-worker / load-balanced options

l = lease.get_lease(LEASE_NAME)
username = os.getenv('USER')
s = server.Server.from_name(f"node-serve-jellyfin-{username}")
s.refresh()
print(f"Server floating IP: {s.floating_ip}")

## Helper functions

In [ ]:
# runs in Chameleon Jupyter environment

def build_image(s, context_dir: str, tag: str):
    print(f"  Building {tag} ...")
    stdout, rc = s.execute(f"docker build -t {tag} {context_dir}")
    if rc != 0:
        raise RuntimeError(f"Build failed:\n{stdout}")
    print(f"  Built: {tag}")


def start_container(s, tag: str, name: str, extra_args: str = ""):
    s.execute(f"docker rm -f {name} 2>/dev/null || true")
    stdout, rc = s.execute(f"docker run -d --rm -p 8000:8000 --name {name} {extra_args} {tag}")
    if rc != 0:
        raise RuntimeError(f"docker run failed:\n{stdout}")
    print(f"  Started: {name}")


def start_compose(s, compose_dir: str, project: str):
    """Start a docker-compose stack (used for onnx_lb)."""
    s.execute(f"docker compose -p {project} -f {compose_dir}/docker-compose.yml down 2>/dev/null || true")
    stdout, rc = s.execute(
        f"docker compose -p {project} -f {compose_dir}/docker-compose.yml up -d --build"
    )
    if rc != 0:
        raise RuntimeError(f"compose up failed:\n{stdout}")
    print(f"  Compose stack '{project}' started")


def wait_healthy(s, name: str, timeout: int = 120):
    print(f"  Waiting for {name} ...", end="", flush=True)
    for _ in range(timeout // 5):
        out, _ = s.execute(f"docker inspect --format='{{{{.State.Health.Status}}}}' {name}")
        if out.strip().strip("'") == "healthy":
            print(" healthy.")
            return
        print(".", end="", flush=True)
        time.sleep(5)
    raise TimeoutError(f"{name} not healthy within {timeout}s")


def wait_compose_healthy(s, project: str, service: str, timeout: int = 120):
    """Wait for a specific compose service container to be healthy."""
    container_name = f"{project}-{service}-1"
    wait_healthy(s, container_name, timeout)


def run_benchmark(s, total: int, concurrency: int, base_url: str = "http://localhost:8000") -> dict:
    cmd = (
        f"cd {REPO_DIR} && "
        f"BASE_URL={base_url} "
        f"TOTAL_REQUESTS={total} CONCURRENCY={concurrency} "
        f"python3 -m benchmarks.benchmark_recommend"
    )
    stdout, rc = s.execute(cmd)
    print(stdout)
    metrics = {}
    for line in stdout.splitlines():
        for key, label in [
            ("Throughput",   "throughput_rps"),
            ("Latency p50",  "p50_ms"),
            ("Latency p95",  "p95_ms"),
            ("Success rate", "success_rate"),
        ]:
            if key in line:
                val = line.split(":")[-1].strip().split()[0].rstrip("%")
                try:
                    metrics[label] = float(val)
                except ValueError:
                    pass
    return metrics


def stop_container(s, name: str):
    s.execute(f"docker rm -f {name} 2>/dev/null || true")
    print(f"  Stopped: {name}")


def stop_compose(s, compose_dir: str, project: str):
    s.execute(f"docker compose -p {project} -f {compose_dir}/docker-compose.yml down")
    print(f"  Compose stack '{project}' stopped")


results = {}
print("Helpers loaded.")

---
## Option 1: baseline
PyTorch `.pt`, 1 uvicorn worker, CPU — **serving reference**

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "baseline"
build_image(s, f"{REPO_DIR}/serving/baseline", "jellyfin-baseline")
start_container(s, "jellyfin-baseline", f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_LOW)
results[OPT] = metrics
stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] {metrics}")

---
## Option 2: onnx_cpu
ONNX Runtime, 1 uvicorn worker, CPU — **model-level optimisation**

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "onnx_cpu"
build_image(s, f"{REPO_DIR}/serving/onnx", "jellyfin-onnx-cpu")
start_container(s, "jellyfin-onnx-cpu", f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_LOW)
results[OPT] = metrics
stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] {metrics}")

---
## Option 3: torch_multiworker
PyTorch `.pt`, 4 gunicorn workers, CPU — **system-level optimisation**

> Same model as baseline, only the serving infrastructure changes (worker count).
> Use high concurrency to actually exercise the extra workers.

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "torch_multiworker"
build_image(s, f"{REPO_DIR}/serving/torch_multiworker", "jellyfin-torch-mw")
start_container(s, "jellyfin-torch-mw", f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_HIGH)
results[OPT] = metrics
stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] {metrics}")

---
## Option 4: onnx_multiworker
ONNX Runtime, 4 gunicorn workers, CPU — **model + system (combined)**

In [ ]:
# runs in Chameleon Jupyter environment
OPT = "onnx_multiworker"
build_image(s, f"{REPO_DIR}/serving/multiworker", "jellyfin-onnx-mw")
start_container(s, "jellyfin-onnx-mw", f"container-{OPT}")
wait_healthy(s, f"container-{OPT}")

metrics = run_benchmark(s, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_HIGH)
results[OPT] = metrics
stop_container(s, f"container-{OPT}")
print(f"\n[{OPT}] {metrics}")

---
## Option 5: onnx_lb
2 × ONNX instances + nginx round-robin, CPU — **infrastructure-level optimisation**

> Horizontal scaling: two independent service containers behind a load balancer.  
> The deployment topology changes — more instances, not just more workers inside one container.

In [ ]:
# runs in Chameleon Jupyter environment
OPT         = "onnx_lb"
COMPOSE_DIR = f"{REPO_DIR}/docker/onnx_lb"
PROJECT     = "jellyfinlb"

start_compose(s, COMPOSE_DIR, PROJECT)
# nginx starts only after both onnx instances are healthy (depends_on condition)
wait_compose_healthy(s, PROJECT, "nginx", timeout=180)

metrics = run_benchmark(s, total=TOTAL_REQUESTS, concurrency=CONCURRENCY_HIGH,
                        base_url="http://localhost:8005")
results[OPT] = metrics
stop_compose(s, COMPOSE_DIR, PROJECT)
print(f"\n[{OPT}] {metrics}")

---
## Results summary table

In [ ]:
# runs in Chameleon Jupyter environment
import pandas as pd

meta = {
    "baseline":          {"opt": "—",                    "hardware": "CPU",  "concurrency": CONCURRENCY_LOW},
    "onnx_cpu":          {"opt": "Model-level",           "hardware": "CPU",  "concurrency": CONCURRENCY_LOW},
    "torch_multiworker": {"opt": "System-level",          "hardware": "CPU",  "concurrency": CONCURRENCY_HIGH},
    "onnx_multiworker":  {"opt": "Model + System",        "hardware": "CPU",  "concurrency": CONCURRENCY_HIGH},
    "onnx_lb":           {"opt": "Infrastructure-level",  "hardware": "CPU×2","concurrency": CONCURRENCY_HIGH},
}

rows = []
for opt, m in results.items():
    rows.append({
        "Option":           opt,
        "Optimisation":     meta[opt]["opt"],
        "Hardware":         meta[opt]["hardware"],
        "Concurrency":      meta[opt]["concurrency"],
        "p50 (ms)":         m.get("p50_ms", "—"),
        "p95 (ms)":         m.get("p95_ms", "—"),
        "Throughput (r/s)": m.get("throughput_rps", "—"),
        "Success rate":     m.get("success_rate", "—"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

---
## Right-sizing: resource usage under load

In [ ]:
# runs in Chameleon Jupyter environment
# Re-start most promising option and snapshot docker stats under load
BEST_OPT    = "onnx_multiworker"   # ← change as needed
BEST_TAG    = "jellyfin-onnx-mw"

start_container(s, BEST_TAG, "container-best")
wait_healthy(s, "container-best")

s.execute(
    f"cd {REPO_DIR} && BASE_URL=http://localhost:8000 "
    f"TOTAL_REQUESTS=300 CONCURRENCY=8 python3 -m benchmarks.benchmark_recommend &"
)
time.sleep(5)
stdout, _ = s.execute("docker stats container-best --no-stream")
print(stdout)
stop_container(s, "container-best")

In [ ]:
# runs in Chameleon Jupyter environment — onnx_lb: check stats for both instances
COMPOSE_DIR = f"{REPO_DIR}/docker/onnx_lb"
PROJECT     = "jellyfinlb"

start_compose(s, COMPOSE_DIR, PROJECT)
wait_compose_healthy(s, PROJECT, "nginx", timeout=180)

s.execute(
    f"cd {REPO_DIR} && BASE_URL=http://localhost:8005 "
    f"TOTAL_REQUESTS=300 CONCURRENCY=8 python3 -m benchmarks.benchmark_recommend &"
)
time.sleep(5)
stdout, _ = s.execute(
    f"docker stats {PROJECT}-onnx_1-1 {PROJECT}-onnx_2-1 --no-stream"
)
print(stdout)
stop_compose(s, COMPOSE_DIR, PROJECT)

---
**Done.** Copy the results table and `docker stats` output into your Gradescope PDF.